Title: “EDA: events, calendar and uncertainty

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/processed/data_daily_base.csv", parse_dates=["date"])

# 1. Weekend/holiday events: move them to the next fixing (otherwise they get lost)
EVENTS = ["is_ecb_decision_day", "is_rbi_decision_day", "is_fomc_decision_day",
          "is_us_cpi_day", "is_us_jobs_day", "is_india_budget_day"]
fix_block = df["eur_inr"].notna()[::-1].cumsum()[::-1]
df[EVENTS] = df.groupby(fix_block)[EVENTS].transform("max")

# 2. Keep only fixing days, from 8 Jan 2020
fx = df[df["eur_inr"].notna() & (df["date"] >= "2020-01-08")].reset_index(drop=True)

# 3. What we study: the daily change of EUR/INR (%) and its size
fx["chg_pct"] = fx["eur_inr"].pct_change() * 100
fx["abs_chg"] = fx["chg_pct"].abs()

# 4. Conclusions only from 2020-2024
train = fx[fx["date"] < "2025-01-01"]

print(fx.shape, train.shape)
print(fx[EVENTS].sum())

(1746, 58) (1279, 58)
is_ecb_decision_day     55
is_rbi_decision_day     42
is_fomc_decision_day    56
is_us_cpi_day           86
is_us_jobs_day          84
is_india_budget_day      8
dtype: int64


## 1. First look at my columns (2020–2024)

In [2]:
MY_COLS = ["gpr", "us_epu", "gpr_india", "in_epu", "eu_epu",
           "weekday", "day_of_month", "month", "is_de_holiday", "is_in_holiday",
           "is_payday_window", "is_diwali", "is_december", "is_india_fy_end",
           *EVENTS, "last_ecb_decision_bp", "last_rbi_decision_bp", "last_fomc_decision_bp"]

print(train[MY_COLS].describe().T[["count", "mean", "min", "max"]].round(2))
print("\nMissing values:", int(train[MY_COLS].isna().sum().sum()))
print(train[["chg_pct", "abs_chg"]].describe().round(3))

                        count    mean     min      max
gpr                    1279.0  115.34    8.32   540.83
us_epu                 1279.0  173.40   10.92  1026.38
gpr_india              1279.0    0.24    0.09     0.51
in_epu                 1279.0   89.74   23.35   193.14
eu_epu                 1279.0  280.29  141.39   479.99
weekday                1279.0    2.00    0.00     4.00
day_of_month           1279.0   15.82    1.00    31.00
month                  1279.0    6.55    1.00    12.00
is_de_holiday          1279.0    0.01    0.00     1.00
is_in_holiday          1279.0    0.05    0.00     1.00
is_payday_window       1279.0    0.14    0.00     1.00
is_diwali              1279.0    0.00    0.00     1.00
is_december            1279.0    0.08    0.00     1.00
is_india_fy_end        1279.0    0.02    0.00     1.00
is_ecb_decision_day    1279.0    0.03    0.00     1.00
is_rbi_decision_day    1279.0    0.02    0.00     1.00
is_fomc_decision_day   1279.0    0.03    0.00     1.00
is_us_cpi_